Import libraries and kaggle csv file 

In [ ]:
import sqlite3
import pandas as pd

db_connection = sqlite3.connect("avishkarna_internship.db")
db_cursor = db_connection.cursor()

csv_file_path = "E-commerce Customer Behavior - Sheet1.csv"
customer_raw_df = pd.read_csv(csv_file_path)

customer_raw_df.to_sql(name="staging_kaggle_data", con=db_connection, if_exists="replace", index=False)
print("Dataset successfully integrated into SQLite Database")

Dataset successfully integrated into SQLite Database!


CREATE TABLE

In [26]:
db_cursor.execute("DROP TABLE IF EXISTS customer_analytics;")

db_cursor.execute("""
CREATE TABLE customer_analytics (
    customer_id INTEGER PRIMARY KEY,                          -- 1. PRIMARY KEY
    gender TEXT NOT NULL,                                      -- 2. NOT NULL
    city TEXT DEFAULT 'Andhra Pradesh',                        -- 3. DEFAULT fallback
    age INTEGER CHECK(age >= 18 AND age <= 100),              -- 4. CHECK constraint
    membership_type TEXT UNIQUE                                -- 5. UNIQUE constraint
);
""")
db_connection.commit()
print("Table created with 5 strict data constraints!")


Table created with 5 strict data constraints!


INSERT

In [27]:
db_cursor.execute("""
INSERT OR IGNORE INTO customer_analytics (customer_id, gender, city, age, membership_type)
SELECT 
    [Customer ID], 
    [Gender], 
    [City], 
    [Age], 
    [Membership Type]
FROM staging_kaggle_data
WHERE [Age] >= 18 AND [Age] <= 100;
""")
db_connection.commit()
print("Kaggle records successfully inserted!")


Kaggle records successfully inserted!


SELECT

In [28]:
query = """
SELECT customer_id, gender, city, age 
FROM customer_analytics 
WHERE age > 25 
LIMIT 5;
"""

filtered_results_df = pd.read_sql_query(query, db_connection)
filtered_results_df


,customer_id,gender,city,age
0,101,Female,New York,29
1,102,Male,Los Angeles,34
2,103,Female,Chicago,43


UPDATE

In [29]:
db_cursor.execute("""
UPDATE customer_analytics 
SET city = 'Vijayawada' 
WHERE customer_id = 1;
""")
db_connection.commit()

pd.read_sql_query("SELECT * FROM customer_analytics WHERE customer_id = 1;", db_connection)


,customer_id,gender,city,age,membership_type


DELETE

In [30]:
db_cursor.execute("""
DELETE FROM customer_analytics 
WHERE age IS NULL OR city = 'Unknown';
""")
db_connection.commit()
print("Target records deleted successfully!")


Target records deleted successfully!


ALTER TABLE

In [31]:
try:
    db_cursor.execute("""
    ALTER TABLE customer_analytics 
    ADD COLUMN ml_cluster_id INTEGER DEFAULT 0;
    """)
    db_connection.commit()
    print("Machine learning feature column successfully added!")
except sqlite3.OperationalError:
    print("Column already exists, skipping schema adjustment.")


Machine learning feature column successfully added!


DROP TABLE

In [33]:
db_cursor.execute("DROP TABLE IF EXISTS staging_kaggle_data;")
db_connection.commit()
print("Temporary workspace environment dropped clean")


Temporary workspace environment dropped clean
